In [1]:
library(tximeta)
library(SummarizedExperiment)

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: 'MatrixGenerics'


The following objects are masked from 'package:matrixStats':

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges, rowRanks,
    rowSdDiffs, rowSds, rowSums2, ro

In [4]:
# 1. Set the exact absolute paths to your local compressed reference files
local_gtf <- "/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/gencode.vM33.annotation.gtf.gz" 
local_fasta <- "/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/gencode.vM33.transcripts.fa.gz" 

# Define directories for the Salmon index and the quantified sample results
index_dir <- "/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/salmon_index_m33"
quant_dir <- "/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/quantified"

cat("Forcing local database linkage to bypass network restrictions...\n")

# 2. Create the linked transcriptome using local files
makeLinkedTxome(
  indexDir = index_dir,
  source = "GENCODE",
  organism = "Mus musculus",
  release = "M33",
  genome = "GRCm39",
  fasta = local_fasta,
  gtf = local_gtf,
  write = FALSE # Set to FALSE to prevent caching errors in the cluster environment
)

# 3. Prepare the sample metadata dataframe for tximeta
sf_files <- list.files(quant_dir, pattern = "quant.sf", recursive = TRUE, full.names = TRUE)
sample_names <- gsub("_quant", "", basename(dirname(sf_files)))
coldata <- data.frame(files = sf_files, names = sample_names, stringsAsFactors = FALSE)

cat("Local linkage complete! Importing quantifications and summarizing to gene level...\n")

# 4. Import the transcript-level data and summarize to the gene level
se <- tximeta(coldata)
gse <- summarizeToGene(se)

# 5. Extract both the raw Counts and the normalized TPM (abundance) matrices
gene_counts <- assay(gse, "counts")
gene_tpm <- assay(gse, "abundance")

# 6. Export both matrices as CSV files to your cluster directory
write.csv(gene_counts, file.path(quant_dir, "tximeta_gene_counts_matrix.csv"))
write.csv(gene_tpm, file.path(quant_dir, "tximeta_gene_tpm_matrix.csv"))

cat("======================================\n")
cat("Success! Both final matrices have been successfully generated.\n")
cat("Counts Matrix dimensions:", nrow(gene_counts), "genes x", ncol(gene_counts), "samples\n")
cat("TPM Matrix dimensions:   ", nrow(gene_tpm), "genes x", ncol(gene_tpm), "samples\n")
cat("======================================\n")

Forcing local database linkage to bypass network restrictions...


reading digest from indexDir: /group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/salmon_index_m33

NOTE: linkedTxome with source='GENCODE', set useHub=FALSE in tximeta
to avoid download of reference txome from AnnotationHub.
alternatively use a different string for source argument, e.g. LocalGENCODE

linkedTxome metadata was same as already in bfc



Local linkage complete! Importing quantifications and summarizing to gene level...


importing salmon quantification files

Warning message:
"'timedatectl' indicates the non-existent timezone name 'n/a'"
Warning message:
"Your system is mis-configured: '/etc/localtime' is not a symlink"
Warning message:
"'/etc/localtime' is not identical to any known timezone file"
reading in files with read.delim ('readr' installed but won't work w/o timezones)

1 
2 
3 
4 
5 
6 
7 
8 
9 
10 
11 
12 
13 
14 
15 
16 
17 


found matching linkedTxome:
[ GENCODE - Mus musculus - release M33 ]

loading existing TxDb created: 2026-07-10 18:18:49

loading existing transcript ranges created: 2026-07-10 18:18:50

fetching genome info for GENCODE

loading existing TxDb created: 2026-07-10 18:18:49

obtaining transcript-to-gene mapping from database

loading existing gene ranges created: 2026-07-10 18:18:55

assignRanges='range': gene ranges assigned by total range of isoforms
  see details at: ?summarizeToGene,SummarizedExperiment-method

summarizing abundance

summarizing counts

summarizing 

Success! Both final matrices have been successfully generated.
Counts Matrix dimensions: 55822 genes x 17 samples
TPM Matrix dimensions:    55822 genes x 17 samples
